In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

S = "workspace.olist_silver_py"
G = "workspace.olist_gold_py"

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{G}.{name}")
    print(name, spark.table(f"{G}.{name}").count())

In [0]:
dates = spark.sql("SELECT explode(sequence(DATE'2016-01-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS full_date")
dim_date = dates.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    "full_date",
    F.year("full_date").alias("year"),
    F.quarter("full_date").alias("quarter"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.dayofmonth("full_date").alias("day_of_month"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"))
save(dim_date, "dim_date")

dim_date 1096


In [0]:
products = spark.table(f"{S}.products").alias("p")
trans = spark.table(f"{S}.category_translation").alias("t")

dim_product = (products
    .join(trans, F.col("p.category_name_pt") == F.col("t.category_name_pt"), "left")
    .select(
        F.row_number().over(Window.orderBy("p.product_id")).alias("product_key"),
        "p.product_id",
        F.coalesce("t.category_name_en", "p.category_name_pt", F.lit("unknown")).alias("category_name"),
        "p.weight_g", "p.length_cm", "p.height_cm", "p.width_cm", "p.photos_qty"))
save(dim_product, "dim_product")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_product 32951


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dim_seller = spark.table(f"{S}.sellers").select(
    F.row_number().over(Window.orderBy("seller_id")).alias("seller_key"),
    "seller_id", "zip_prefix", "city", "state")
save(dim_seller, "dim_seller")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_seller 3095


In [0]:
cust = spark.table(f"{S}.customers")
orders = spark.table(f"{S}.orders")

co = (cust.join(orders, "customer_id")
          .select("customer_unique_id", "customer_id", "zip_prefix", "city", "state", "purchased_at"))

w_order = Window.partitionBy("customer_unique_id").orderBy("purchased_at", "customer_id")

flagged = co.withColumn(
    "is_change",
    F.when((F.col("city") == F.lag("city").over(w_order)) &
           (F.col("state") == F.lag("state").over(w_order)), 0).otherwise(1))

numbered = flagged.withColumn(
    "version_no",
    F.sum("is_change").over(w_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)))

versions = (numbered.groupBy("customer_unique_id", "version_no")
    .agg(F.min("zip_prefix").alias("zip_prefix"),
         F.min("city").alias("city"),
         F.min("state").alias("state"),
         F.min("purchased_at").alias("valid_from")))

w_ver = Window.partitionBy("customer_unique_id").orderBy("version_no")

dim_customer = versions.select(
    F.row_number().over(Window.orderBy("customer_unique_id", "version_no")).alias("customer_key"),
    "customer_unique_id", "zip_prefix", "city", "state", "valid_from",
    F.lead("valid_from").over(w_ver).alias("valid_to"),
    F.lead("valid_from").over(w_ver).isNull().alias("is_current"))
save(dim_customer, "dim_customer")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customer 96220


In [0]:
oi = spark.table(f"{S}.order_items").alias("oi")
o = spark.table(f"{S}.orders").alias("o")
c = spark.table(f"{S}.customers").alias("c")
dc = spark.table(f"{G}.dim_customer").alias("dc")
dp = spark.table(f"{G}.dim_product").alias("dp")
ds = spark.table(f"{G}.dim_seller").alias("ds")

fact = (oi
    .join(o, F.col("oi.order_id") == F.col("o.order_id"))
    .join(c, F.col("o.customer_id") == F.col("c.customer_id"))
    .join(dc,
          (F.col("dc.customer_unique_id") == F.col("c.customer_unique_id")) &
          (F.col("o.purchased_at") >= F.col("dc.valid_from")) &
          (F.col("dc.valid_to").isNull() | (F.col("o.purchased_at") < F.col("dc.valid_to"))),
          "left")
    .join(dp, F.col("dp.product_id") == F.col("oi.product_id"), "left")
    .join(ds, F.col("ds.seller_id") == F.col("oi.seller_id"), "left")
    .select(
        F.col("oi.order_id").alias("order_id"),
        "oi.order_item_id",
        "dc.customer_key", "dp.product_key", "ds.seller_key",
        F.date_format("o.purchased_at", "yyyyMMdd").cast("int").alias("purchase_date_key"),
        F.date_format("o.delivered_at", "yyyyMMdd").cast("int").alias("delivered_date_key"),
        "o.order_status",
        "oi.price", "oi.freight_value",
        (F.col("oi.price") + F.col("oi.freight_value")).alias("total_item_value"),
        F.datediff("o.delivered_at", "o.purchased_at").alias("delivery_days"),
        F.datediff("o.delivered_at", "o.estimated_delivery_at").alias("days_vs_estimate")))
save(fact, "fact_order_items")

fact_order_items 112650


In [0]:
f = spark.table(f"{G}.fact_order_items")
f.agg(F.count("*").alias("fact_rows"),
      F.count("customer_key").alias("with_customer"),
      F.count("product_key").alias("with_product"),
      F.count("seller_key").alias("with_seller")).show()

+---------+-------------+------------+-----------+
|fact_rows|with_customer|with_product|with_seller|
+---------+-------------+------------+-----------+
|   112650|       112650|      112650|     112650|
+---------+-------------+------------+-----------+



In [0]:
tables = ["dim_date", "dim_product", "dim_seller", "dim_customer", "fact_order_items"]
for t in tables:
    a = spark.table(f"workspace.olist_gold_sql.{t}").count()
    b = spark.table(f"workspace.olist_gold_py.{t}").count()
    print(f"{t:20} sql={a:>7}  py={b:>7}  {'OK' if a == b else 'MISMATCH'}")

q = "SELECT ROUND(SUM(price), 2), ROUND(SUM(freight_value), 2) FROM workspace.olist_gold_{}.fact_order_items"
print("revenue and freight, sql:", spark.sql(q.format("sql")).first())
print("revenue and freight, py: ", spark.sql(q.format("py")).first())

dim_date             sql=   1096  py=   1096  OK
dim_product          sql=  32951  py=  32951  OK
dim_seller           sql=   3095  py=   3095  OK
dim_customer         sql=  96220  py=  96220  OK
fact_order_items     sql= 112650  py= 112650  OK
revenue and freight, sql: Row(ROUND(SUM(price),2)=Decimal('13591643.70'), ROUND(SUM(freight_value),2)=Decimal('2251909.54'))
revenue and freight, py:  Row(ROUND(SUM(price),2)=Decimal('13591643.70'), ROUND(SUM(freight_value),2)=Decimal('2251909.54'))
